# 07 · Durability, publisher confirms, and unroutable messages

**Level:** intermediate · **Time:** 35 minutes · **Before you start:** complete setup 00 and understand acknowledgements from 02.

By the end you can distinguish queue durability, message persistence, publisher confirms, mandatory routing, and consumer acknowledgements. You will deliberately publish an unroutable message and catch it.

| Mechanism | What it addresses |
|---|---|
| Durable queue | Queue definition survives a broker restart |
| Persistent message (`delivery_mode=2`) | Request storage of the message when routed to a suitable durable queue |
| Publisher confirm | Broker accepts responsibility for a publication; does not mean the consumer finished |
| `mandatory=True` | Report when a publication cannot route to any queue |
| Consumer acknowledgement | Consumer tells the broker it finished processing |

**Prediction:** does a durable queue alone make transient messages durable? No. A confirm alone also does not prove that any queue matched: add mandatory routing when that matters.

Run cells in order. Only the explicitly marked terminal restart is optional.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course" / "rabbit.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.rabbit import connect, unique_name, wait_for_message, broker_url
import pika
print("Course root:", ROOT)


## Step 1 — Declare durable storage and confirm a persistent publication

The default exchange routes by queue name. We give the queue a unique name so another lesson cannot consume its message. The `with` block closes the connection; closing a connection does not remove this durable, non-exclusive queue.


In [ ]:
queue_name = unique_name("durable")
payload = b"invoice-1001: created"
with connect() as connection:
    channel = connection.channel()
    channel.queue_declare(queue=queue_name, durable=True)
    channel.confirm_delivery()
    channel.basic_publish(
        exchange="", routing_key=queue_name, body=payload,
        properties=pika.BasicProperties(delivery_mode=2, content_type="text/plain"),
        mandatory=True,
    )
print("Broker confirmed publication to", queue_name)


## Optional pause — Test a real broker restart

To demonstrate persistence, **pause before Step 2**, run `docker compose restart rabbitmq` in a terminal at the course root, and wait for `docker compose ps` to show RabbitMQ healthy. Keep the Docker volume; do not run `down -v`.

Without this manual step, the next assertion proves survival across a client disconnect only. A single broker and disk are still a single failure domain. Quorum replication is covered in 17.


## Step 2 — Reconnect, read, and acknowledge

Expected: the body is unchanged and its persistence property is `2`. Acknowledge only after checking the result.


In [ ]:
with connect() as connection:
    channel = connection.channel()
    method, properties, body = wait_for_message(channel, queue_name)
    assert body == payload
    assert properties.delivery_mode == 2
    channel.basic_ack(method.delivery_tag)
print("Received and acknowledged:", body.decode())


## Step 3 — Fail visibly when no queue matches

We intentionally publish to a queue name that was never declared. In Pika's blocking confirm mode, an unroutable mandatory publication raises `UnroutableError`. Treat a connection failure during publishing differently: the outcome can be unknown, so a retry may duplicate the message.


In [ ]:
with connect() as connection:
    channel = connection.channel()
    channel.confirm_delivery()
    try:
        channel.basic_publish(exchange="", routing_key=unique_name("missing"),
                              body=b"must not vanish silently", mandatory=True)
    except pika.exceptions.UnroutableError:
        print("Expected: RabbitMQ returned the unroutable message.")
    else:
        raise AssertionError("Expected an unroutable message error")


## Step 4 — Observe an incompatible declaration

An existing durable queue cannot be redeclared as non-durable. The broker closes the **channel**, so isolate the deliberate error on its own channel and connection. Expected AMQP reply code: `406 PRECONDITION_FAILED`.


In [ ]:
with connect() as connection:
    channel = connection.channel()
    try:
        channel.queue_declare(queue=queue_name, durable=False)
    except pika.exceptions.ChannelClosedByBroker as error:
        assert error.reply_code == 406
        print("Expected declaration mismatch:", error.reply_code)
    else:
        raise AssertionError("Incompatible declaration unexpectedly succeeded")


## Exercise — Explain the two failure windows

1. The broker stores a message, but the connection dies before the publisher sees its confirm. Should the publisher retry?
2. The consumer commits a database change, then dies before acknowledging. What happens on redelivery?

**Suggested answer:** retry when delivery is required, keeping the same event ID, and make the consumer idempotent. Both windows can produce duplicates. Publisher confirms and acknowledgements do not provide end-to-end exactly-once execution. Notebook 09 makes the database side effect safe to repeat.

**Troubleshooting:** a `406` in normal code usually means mismatched queue/exchange arguments. Use a new lesson queue name or reconcile its definition. A confirm timeout is an uncertain outcome, not evidence that the broker discarded the message.


## Cleanup — Delete only this notebook's queue


In [ ]:
with connect() as connection:
    connection.channel().queue_delete(queue=queue_name)
print("Deleted", queue_name)


## References and next step

- [RabbitMQ acknowledgements and confirms](https://www.rabbitmq.com/docs/4.2/confirms)
- [RabbitMQ queue durability](https://www.rabbitmq.com/docs/4.2/queues#durability)
- [Pika delivery confirmation example](https://pika.readthedocs.io/en/stable/examples/blocking_delivery_confirmations.html)

Next: give failed messages a bounded retry path in 08.
